# EDA - entidad Prestamo

Caracteristicas de la transaccion financiera (monto, tasa, proposito, resultado).

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

COLS = ['loan_amnt', 'loan_intent', 'loan_int_rate', 'loan_percent_income', 'loan_status']

df_full = pd.read_csv(ROOT / 'data' / 'loan_data.csv')
pre = df_full[COLS].copy()
pre['person_income_join'] = df_full['person_income']  # para ratios derivados
pre.shape

## Vista general

In [ ]:
pre.head()

In [ ]:
pre.describe(include='all').T

## Distribucion por proposito (`loan_intent`)

In [ ]:
pre.groupby('loan_intent').agg(
    n=('loan_status', 'size'),
    tasa_default=('loan_status', 'mean'),
    monto_promedio=('loan_amnt', 'mean'),
    tasa_promedio=('loan_int_rate', 'mean'),
).round(4).sort_values('tasa_default', ascending=False)

## Outliers y violaciones de reglas

In [ ]:
violaciones = {
    'loan_int_rate fuera de [5, 30]':      int((~pre['loan_int_rate'].between(5, 30)).sum()),
    'loan_percent_income fuera de [0, 1]': int((~pre['loan_percent_income'].between(0, 1)).sum()),
    'loan_amnt <= 0':                      int((pre['loan_amnt'] <= 0).sum()),
}
pd.DataFrame.from_dict(violaciones, orient='index', columns=['n_violaciones'])

In [ ]:
pd.DataFrame({
    'loan_amnt': pre['loan_amnt'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2),
    'loan_int_rate': pre['loan_int_rate'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2),
})

## Que variables del prestamo predicen default

In [ ]:
num = pre.select_dtypes(include='number').drop(columns=['person_income_join'])
num.corr()['loan_status'].drop('loan_status').abs().sort_values(ascending=False).to_frame('|corr|').round(4)

## Hallazgo

- `loan_percent_income` (0.38), `loan_int_rate` (0.33) y `loan_intent` (spread 0.16) son los predictores reales del prestamo.
- `loan_amnt` solo tiene |corr| 0.11 — su poder predictivo aparece cuando se combina con ingreso (ver `features.ipynb`).